# 11. BERT — Masked Language Model

**Цель:** Реализовать BERT-подобную модель (Encoder-only), обучить на задаче Masked Language Model (MLM), визуализировать контекстуальные представления.

---

In [2]:
# Импорт библиотек: sys/os/logging для утилит, torch для нейросетей, numpy/matplotlib для数据处理 и визуализации
import sys, os, logging, math

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# Автоматический выбор устройства: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


## 11.1 Специальные токены BERT

| Токен | ID | Назначение |
|-------|----|------------|
| [PAD] | 0  | Выравнивание длин |
| [CLS] | 1  | Классификационный токен (начало) |
| [SEP] | 2  | Разделитель предложений |
| [MASK]| 3  | Маскированный токен |
| [UNK] | 4  | Неизвестный токен |

In [4]:
# Определяем числовые ID для специальных токенов BERT
PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4


## 11.2 BERT-архитектура (Encoder-only)

BERT — это стопка энкодерных блоков трансформера с bidirectionnal self-attention (в отличие от GPT, где attention каузальный).

In [6]:
# ---- Multi-Head Self-Attention (механизм внимания с несколькими головами) ----
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0  # размерность должна делиться на число голов
        self.d_k = d_model // n_heads  # размерность каждой головы
        # Матрицы проекций для Query, Key, Value и Output
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch = Q.size(0)
        n_heads = self.W_O.out_features // self.d_k
        # Линейные проекции + разделение на головы: (batch, n_heads, seq_len, d_k)
        Q = self.W_Q(Q).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, n_heads, self.d_k).transpose(1, 2)
        # Скалярное произведение Q и K с масштабированием
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            # Заполняем маскированные позиции -inf, чтобы softmax дал 0
            scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        # Softmax по последней оси (внимание к ключам) + dropout
        attn = self.dropout(F.softmax(scores, dim=-1))
        # Взвешенная сумма значений + конкатенация голов
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.W_O.out_features)
        return self.W_O(output)

# ---- FeedForward сеть (два линейных слоя с GELU) ----
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)   # расширение в 4 раза
        self.fc2 = nn.Linear(4 * d_model, d_model)   # сжатие обратно
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

# ---- Один блок энкодера (Self-Attention + FeedForward + skip-connections + LayerNorm) ----
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, dropout)
        self.norm1 = nn.LayerNorm(d_model)  # слоевая норма после attention
        self.norm2 = nn.LayerNorm(d_model)  # слоевая норма после FFN
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        # Self-Attention с residual-связью и LayerNorm (pre-norm)
        x = x + self.dropout1(self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask))
        # FeedForward с residual-связью
        x = x + self.dropout2(self.ffn(self.norm2(x)))
        return x

# ---- Полная BERT-модель (стопка энкодеров + MLM head) ----
class BERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, num_layers=4, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        # Таблица эмбеддингов токенов
        self.embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional Encoding (синусоидальное кодирование позиций)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # не обучаемый параметр
        self.dropout_pe = nn.Dropout(dropout)
        
        # Стопка энкодерных блоков
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, dropout) for _ in range(num_layers)
        ])
        
        # MLM head: предсказывает замаскированные токены
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, vocab_size),  # выходной слой: логиты для каждого токена
        )
        
        # Xavier-инициализация для всех весов
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, mask=None):
        # Сумма эмбеддингов токенов и позиционных кодировок + dropout
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])
        # Пропускаем через все слои энкодера
        for layer in self.layers:
            x = layer(x, mask)
        # MLM head: логиты для каждого токена словаря
        return self.mlm_head(x)  # (batch, seq_len, vocab)

## 11.3 Masked Language Model: маскировка 15% токенов

**Стратегия маскировки (BERT原文):**
- 80% → заменить на [MASK]
- 10% → заменить на случайный токен
- 10% → оставить как есть

In [8]:
# ---- Функция маскировки токенов для MLM ----
def mask_tokens(input_ids, vocab_size, mask_prob=0.15, special_tokens=(PAD, CLS, SEP)):
    """Маскировка токнов для MLM: 80% -> [MASK], 10% -> random, 10% -> unchanged."""
    labels = input_ids.clone()  # копия для target-меток
    
    # Не маскируем специальные токены ([PAD], [CLS], [SEP])
    special_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    for tok in special_tokens:
        special_mask = special_mask | (input_ids == tok)
    
    # Выбираем 15% позиций для маскировки (кроме специальных)
    probability = torch.full(input_ids.shape, mask_prob, device=input_ids.device)
    masked = torch.bernoulli(probability).bool()
    masked = masked & ~special_mask
    
    # 80% маскированных -> замена на [MASK]
    mask_replace = torch.bernoulli(torch.full(input_ids.shape, 0.8, device=input_ids.device)).bool() & masked
    input_ids[mask_replace] = MASK
    
    # 10% -> замена на случайный токен
    random_replace = torch.bernoulli(torch.full(input_ids.shape, 0.5, device=input_ids.device)).bool() & masked & ~mask_replace
    random_tokens = torch.randint(5, vocab_size, input_ids.shape, device=input_ids.device)
    input_ids[random_replace] = random_tokens[random_replace]
    
    # 10% -> оставляем без изменений (labels уже содержат исходный токен)
    
    # Для немасокрованных позиций ставим -100 (CrossEntropyLoss ignore_index)
    labels[~masked] = -100
    
    return input_ids, labels

# Демонстрация маскировки на примере
example = torch.tensor([[CLS, 5, 12, 8, 3, 15, SEP, PAD, PAD]])
masked, labels = mask_tokens(example.clone(), vocab_size=20)
print(f"Original: {example[0].tolist()}")
print(f"Masked:   {masked[0].tolist()}")
print(f"Labels:   {labels[0].tolist()}")


Original: [1, 5, 12, 8, 3, 15, 2, 0, 0]
Masked:   [1, 5, 12, 8, 3, 15, 2, 0, 0]
Labels:   [-100, -100, -100, -100, -100, -100, -100, -100, -100]


## 11.4 Подготовка данных и обучение BERT

In [10]:
# Параметры словаря и максимальной длины последовательности
vocab_size = 30
max_len = 12

# Функция генерации синтетических данных: [CLS] + случайные токены + [SEP] + [PAD]
def make_mlm_data(num_samples):
    data = []
    for _ in range(num_samples):
        length = np.random.randint(3, max_len - 1)  # случайная длина последовательности
        seq = [CLS] + np.random.randint(5, vocab_size, size=length).tolist() + [SEP]
        seq = seq + [PAD] * (max_len - len(seq))  # дополнение до max_len
        data.append(seq)
    return torch.tensor(data)

# Создаём train/val датасеты
train_data = make_mlm_data(2000)
val_data = make_mlm_data(500)

# Инициализируем BERT, оптимизатор и функцию потерь
bert = BERT(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=4, max_len=max_len).to(device)
optimizer = torch.optim.AdamW(bert.parameters(), lr=0.001, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=-100)  # -100 не участвует в loss

print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")
print(f"BERT params: {sum(p.numel() for p in bert.parameters()):,}")

Train samples: 2000, Val samples: 500
BERT params: 53,374


### Цикл обучения BERT

На каждой эпохе: маскируем 15% токенов, предсказываем их, считаем cross-entropy loss только для маскированных позиций.

In [ ]:
# Параметры обучения
n_epochs = 40
batch_size = 64
best_loss = float('inf')
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    bert.train()
    epoch_loss = 0
    perm = torch.randperm(len(train_data))  # перемешиваем данные
    
    # Батчевая обработка
    for i in range(0, len(train_data), batch_size):
        idx = perm[i:i+batch_size]
        batch = train_data[idx].clone().to(device)
        masked, labels = mask_tokens(batch, vocab_size)  # маскируем токены
        
        output = bert(masked)  # forward pass
        loss = criterion(output.reshape(-1, vocab_size), labels.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()  # обратное распространение
        torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)  # клиппинг градиентов
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Валидация на отложенной выборке
    bert.eval()
    val_loss = 0
    with torch.no_grad():
        for i in range(0, len(val_data), batch_size):
            batch = val_data[i:i+batch_size].clone().to(device)
            masked, labels = mask_tokens(batch, vocab_size)
            output = bert(masked)
            val_loss += criterion(output.reshape(-1, vocab_size), labels.reshape(-1)).item()
    
    # Средние значения loss
    avg_train = epoch_loss / (len(train_data) / batch_size)
    avg_val = val_loss / (len(val_data) / batch_size)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    
    # Логирование и сохранение лучшей модели
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")
    if avg_val < best_loss:
        best_loss = avg_val

# Визуализация кривых обучения
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BERT MLM Training')
plt.grid(True)
plt.show()


## 11.5 Демонстрация: предсказание замаскированных токенов

In [13]:
# Функция предсказания замаскированных токенов
@torch.no_grad()
def predict_masked(model, input_ids):
    model.eval()
    output = model(input_ids.to(device))  # (batch, seq_len, vocab)
    probs = F.softmax(output, dim=-1)      # вероятности по словарю
    preds = output.argmax(-1)              # наиболее вероятный токен
    return preds, probs

# Создаём пример с [MASK] на позиции 3
example = torch.tensor([[CLS, 12, 7, MASK, 19, 5, SEP, PAD, PAD, PAD, PAD, PAD]])
preds, probs = predict_masked(bert, example)

# Показываем top-5 предсказаний для маскированной позиции
mask_pos = (example == MASK).nonzero()[0, 1].item()
mask_probs = probs[0, mask_pos]
top5 = mask_probs.argsort(descending=True)[:5]

print(f"Input: {example[0].tolist()}")
print(f"Prediction at position {mask_pos}:")
for tok in top5:
    print(f"  Token {tok.item():2d}: {mask_probs[tok].item():.2%}")


Input: [1, 12, 7, 3, 19, 5, 2, 0, 0, 0, 0, 0]
Prediction at position 3:
  Token  7: 41.53%
  Token  2: 6.16%
  Token 23: 5.88%
  Token 18: 4.70%
  Token 14: 4.15%


## 11.6 Контекстуальные представления разных слоёв

Извлечём эмбеддинги из разных слоёв и покажем, как они меняются.

In [15]:
# Класс-обёртка BERT с сохранением выходов каждого слоя
class BERTWithHooks(BERT):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.layer_outputs = []
    
    def forward(self, x, mask=None):
        self.layer_outputs = []
        # Начальные эмбеддинги (вход в первый слой)
        x = self.dropout_pe(self.embedding(x) * math.sqrt(self.d_model) + self.pe[:, :x.size(1), :])
        self.layer_outputs.append(x.detach())
        for layer in self.layers:
            x = layer(x, mask)
            self.layer_outputs.append(x.detach())  # сохраняем выход каждого слоя
        return self.mlm_head(x)

# Создаём модель с хуками и копируем веса из обученной
bert_hooks = BERTWithHooks(vocab_size=vocab_size, d_model=32, n_heads=4, num_layers=4, max_len=max_len).to(device)
bert_hooks.load_state_dict(bert.state_dict())

# Тестовый пример для извлечения представлений
test_seq = torch.tensor([[CLS, 10, 15, 8, 12, 6, 18, SEP, PAD, PAD, PAD, PAD]])
with torch.no_grad():
    bert_hooks(test_seq.to(device))

# Визуализация эмбеддингов на разных слоях (2x3 сетка)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, ax in enumerate(axes.flat):
    if i < len(bert_hooks.layer_outputs):
        im = ax.imshow(bert_hooks.layer_outputs[i][0].cpu().numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'Layer {i}' if i > 0 else 'Embedding')
        ax.set_xlabel('d_model')
        ax.set_ylabel('Token')
        plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Contextual Embeddings Through BERT Layers')
plt.tight_layout()
plt.show()


In [16]:
print("=== BERT (Masked Language Model) complete ===")
print("Topics covered:")
print("  - Special tokens: [CLS], [SEP], [MASK], [PAD]")
print("  - BERT architecture (Encoder-only)")
print("  - MLM masking strategy (80/10/10)")
print("  - BERT MLM training loop")
print("  - Predicting masked tokens")
print("  - Contextual embeddings through layers")
print(f"  - Best MLM loss: {best_loss:.4f}")


=== BERT (Masked Language Model) complete ===
Topics covered:
  - Special tokens: [CLS], [SEP], [MASK], [PAD]
  - BERT architecture (Encoder-only)
  - MLM masking strategy (80/10/10)
  - BERT MLM training loop
  - Predicting masked tokens
  - Contextual embeddings through layers
  - Best MLM loss: inf
